# 01 - ETL: Load Raw CMS DE-SynPUF Files

In [ ]:
# load config and set up paths
import yaml
from pathlib import Path
import pandas as pd

with open("../configs/data_sources.yaml") as f:
    config = yaml.safe_load(f)

raw_dir = Path("..") / config["raw_dir"]
interim_dir = Path("..") / config["interim_dir"]
interim_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# helper to load and concat all files matching a glob pattern
def load_csv_glob(directory, pattern):
    frames = [pd.read_csv(p, low_memory=False) for p in sorted(directory.glob(pattern))]
    if not frames:
        raise FileNotFoundError(f"no files found for pattern: {pattern} in {directory}")
    return pd.concat(frames, ignore_index=True)


In [ ]:
# load each raw file and write it out as parquet in the interim layer
# CMS ships column headers in ALL CAPS -- lowercased here once so every sql
# file downstream can just use lowercase names without re-aliasing everything
for name, spec in config["files"].items():
    df = load_csv_glob(raw_dir, spec["pattern"])
    df.columns = df.columns.str.lower()
    out_path = interim_dir / f"{name}.parquet"
    df.to_parquet(out_path, index=False)
    print(f"wrote {out_path} ({len(df):,} rows, {df.shape[1]} cols)")


In [ ]:
# quick peek at one beneficiary summary year to sanity check the load
# (there are 3 separate beneficiary files, one per year -- see the theory doc
# for why they're kept separate instead of concatenated into one table)
pd.read_parquet(interim_dir / "beneficiary_summary_2009.parquet").head()
